# 🔍 Exploratory Data Analysis — Sparkov Fraud Detection Dataset

**Mục tiêu**: Khám phá và hiểu bộ dữ liệu Sparkov trước khi tiến hành tiền xử lý và huấn luyện mô hình.  
**Phạm vi**: Chỉ EDA — KHÔNG encoding, KHÔNG xử lý mất cân bằng, KHÔNG train model.

---

## 0. Setup & Imports

In [49]:
import sys
from pathlib import Path

# Thêm project root vào sys.path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from src.config import (
    RAW_DATA_DIR, FIGURES_DIR, PROFILING_DIR,
    TRAIN_FILE, TEST_FILE, TARGET_COL,
    NUMERICAL_COLS, CATEGORICAL_COLS,
)

# Cấu hình hiển thị
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# Style biểu đồ
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['savefig.bbox'] = 'tight'

# Hàm lưu biểu đồ
def save_fig(name: str, fig=None):
    """Lưu figure hiện tại vào reports/figures/ dưới dạng PNG."""
    path = FIGURES_DIR / f"{name}.png"
    if fig is not None:
        fig.savefig(path, bbox_inches='tight', facecolor='white')
    else:
        plt.savefig(path, bbox_inches='tight', facecolor='white')
    print(f"💾 Đã lưu: {path}")

print(f"📁 Project root: {PROJECT_ROOT}")
print(f"📂 Raw data dir: {RAW_DATA_DIR}")
print(f"📊 Figures dir:  {FIGURES_DIR}")
print("\n✅ Setup hoàn tất!")

📁 Project root: /Users/thetrung/Projects/Fraud Detection - CIES
📂 Raw data dir: /Users/thetrung/Projects/Fraud Detection - CIES/data/raw
📊 Figures dir:  /Users/thetrung/Projects/Fraud Detection - CIES/reports/figures

✅ Setup hoàn tất!


---
## 1. Load dữ liệu & Thông tin tổng quan

Đọc file `fraudTrain.csv` và `fraudTest.csv` của bộ Sparkov.  
Kiểm tra kích thước, kiểu dữ liệu, missing values, và duplicate rows.

In [50]:
# Load dữ liệu
df_train = pd.read_csv(RAW_DATA_DIR / TRAIN_FILE).drop(columns=['Unnamed: 0'], errors='ignore')
df_test = pd.read_csv(RAW_DATA_DIR / TEST_FILE).drop(columns=['Unnamed: 0'], errors='ignore')

print(f"🔹 Train set: {df_train.shape[0]:,} rows × {df_train.shape[1]} columns")
print(f"🔹 Test set:  {df_test.shape[0]:,} rows × {df_test.shape[1]} columns")
print(f"\n🔹 Tổng cộng: {df_train.shape[0] + df_test.shape[0]:,} giao dịch")

🔹 Train set: 1,296,675 rows × 23 columns
🔹 Test set:  555,719 rows × 23 columns

🔹 Tổng cộng: 1,852,394 giao dịch


In [51]:
# Gộp train + test để EDA toàn diện (đánh dấu nguồn)
df_train['_source'] = 'train'
df_test['_source'] = 'test'
df = pd.concat([df_train, df_test], ignore_index=True)

print(f"📦 Dataset gộp: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\n🔹 Các cột: {list(df.columns)}")

📦 Dataset gộp: 1,852,394 rows × 24 columns

🔹 Các cột: ['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud', '_source']


In [52]:
# Xem 5 dòng đầu
df.head()

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,_source
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.9700,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.0113,-82.0483,0,train
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.2300,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.1590,-118.1865,0,train
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.1100,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,ID,83252,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.1507,-112.1545,0,train
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.0000,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,MT,59632,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.0343,-112.5611,0,train
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.9600,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,VA,24433,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.6750,-78.6325,0,train


In [53]:
# Thông tin tổng quan
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1852394 entries, 0 to 1852393
Data columns (total 24 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Unnamed: 0             int64  
 1   trans_date_trans_time  object 
 2   cc_num                 int64  
 3   merchant               object 
 4   category               object 
 5   amt                    float64
 6   first                  object 
 7   last                   object 
 8   gender                 object 
 9   street                 object 
 10  city                   object 
 11  state                  object 
 12  zip                    int64  
 13  lat                    float64
 14  long                   float64
 15  city_pop               int64  
 16  job                    object 
 17  dob                    object 
 18  trans_num              object 
 19  unix_time              int64  
 20  merch_lat              float64
 21  merch_long             float64
 22  is_fraud          

### 1.1 Kiểm tra Missing Values

In [54]:
# Bảng missing values
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(4)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print("📋 Tỷ lệ Missing Values:")
display(missing_df)

if missing.sum() == 0:
    print("\n✅ Không có missing values trong dataset!")
else:
    print(f"\n⚠️ Tổng missing values: {missing.sum():,}")

📋 Tỷ lệ Missing Values:


,Missing Count,Missing %
Unnamed: 0,0,0.0000
trans_date_trans_time,0,0.0000
is_fraud,0,0.0000
merch_long,0,0.0000
merch_lat,0,0.0000
unix_time,0,0.0000
trans_num,0,0.0000
dob,0,0.0000
job,0,0.0000
city_pop,0,0.0000



✅ Không có missing values trong dataset!


### 1.2 Kiểm tra Duplicate Rows

In [ ]:
# Kiểm tra duplicate (bỏ cột _source và Unnamed: 0 nếu có)
check_cols = [c for c in df.columns if c not in ['_source', 'Unnamed: 0']]
n_dup = df.duplicated(subset=check_cols).sum()
print(f"🔍 Số duplicate rows: {n_dup:,}")
print(f"   Tỷ lệ: {n_dup / len(df) * 100:.4f}%")

if n_dup == 0:
    print("\n✅ Không có duplicate rows!")

---
## 2. Thống kê mô tả

Phân tích thống kê cơ bản cho các biến numerical và categorical.

### 2.1 Thống kê Numerical

In [ ]:
# Thống kê mô tả cho các cột numerical
num_cols_in_df = [c for c in NUMERICAL_COLS if c in df.columns]
print(f"📊 Thống kê mô tả cho {len(num_cols_in_df)} cột numerical:\n")
display(df[num_cols_in_df].describe().T.style.format('{:.2f}'))

### 2.2 Thống kê Categorical

Top 15 giá trị phổ biến nhất cho mỗi cột categorical, kèm tổng số category duy nhất (cardinality).

In [ ]:
cat_cols_in_df = [c for c in CATEGORICAL_COLS if c in df.columns]

for col in cat_cols_in_df:
    n_unique = df[col].nunique()
    print(f"\n{'='*60}")
    print(f"📌 {col} — Cardinality: {n_unique:,} giá trị duy nhất")
    print(f"{'='*60}")
    
    top_15 = df[col].value_counts().head(15)
    top_15_df = pd.DataFrame({
        'Count': top_15.values,
        '% of Total': (top_15.values / len(df) * 100).round(2)
    }, index=top_15.index)
    display(top_15_df)

print(f"\n\n📋 Tóm tắt Cardinality:")
card_summary = pd.DataFrame({
    'Column': cat_cols_in_df,
    'Unique Values': [df[c].nunique() for c in cat_cols_in_df]
}).sort_values('Unique Values', ascending=False)
display(card_summary)

---
## 3. Phân tích mất cân bằng (Class Imbalance)

Kiểm tra tỷ lệ giữa giao dịch hợp lệ (`is_fraud=0`) và gian lận (`is_fraud=1`).  
⚠️ **Chỉ mô tả, chưa xử lý** — việc xử lý mất cân bằng sẽ thực hiện ở bước tiền xử lý.

In [ ]:
from IPython.core.display_functions import display
# Tỷ lệ is_fraud
fraud_counts = df[TARGET_COL].value_counts()
fraud_pct = df[TARGET_COL].value_counts(normalize=True) * 100

fraud_summary = pd.DataFrame({
    'Label': ['Hợp lệ (0)', 'Gian lận (1)'],
    'Count': fraud_counts.values,
    'Percentage (%)': fraud_pct.values.round(4)
})
display(fraud_summary)

# Class imbalance ratio
n_legit = fraud_counts[0]
n_fraud = fraud_counts[1]
imbalance_ratio = n_legit / n_fraud
print(f"\n⚖️ Class Imbalance Ratio: {imbalance_ratio:.1f} : 1 (legit : fraud)")
print(f"   Cứ ~{imbalance_ratio:.0f} giao dịch hợp lệ thì có 1 giao dịch gian lận.")

In [ ]:
# Bar chart class imbalance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
colors = ['#2ecc71', '#e74c3c']
bars = axes[0].bar(
    ['Hợp lệ (0)', 'Gian lận (1)'],
    fraud_counts.values,
    color=colors, edgecolor='black', linewidth=0.5
)
for bar, count in zip(bars, fraud_counts.values):
    axes[0].text(
        bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
        f'{count:,}', ha='center', va='bottom', fontweight='bold', fontsize=11
    )
axes[0].set_title('Số lượng giao dịch theo lớp', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Số giao dịch')

# Pie chart
axes[1].pie(
    fraud_counts.values, labels=['Hợp lệ (0)', 'Gian lận (1)'],
    autopct='%1.2f%%', colors=colors,
    explode=(0, 0.1), shadow=True, startangle=140,
    textprops={'fontsize': 11}
)
axes[1].set_title('Tỷ lệ phân bố lớp', fontsize=13, fontweight='bold')

fig.suptitle('Phân tích mất cân bằng — is_fraud', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('01_class_imbalance', fig)
plt.show()

---
## 4. Trực quan hoá phân phối

Phân tích phân phối các biến chính, đặc biệt so sánh giữa giao dịch hợp lệ và gian lận.

### 4.1 Phân phối số tiền giao dịch (`amt`)

So sánh phân phối `amt` tổng thể và tách theo `is_fraud`.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Histogram toàn bộ
axes[0, 0].hist(df['amt'], bins=100, color='steelblue', edgecolor='white', alpha=0.8)
axes[0, 0].set_title('Phân phối amt (toàn bộ)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Số tiền (amt)')
axes[0, 0].set_ylabel('Tần suất')

# 2. KDE tách theo is_fraud
for label, color, name in [(0, '#2ecc71', 'Hợp lệ'), (1, '#e74c3c', 'Gian lận')]:
    subset = df[df[TARGET_COL] == label]['amt']
    axes[0, 1].hist(subset, bins=100, alpha=0.6, color=color, label=name, density=True)
axes[0, 1].set_title('Phân phối amt — tách theo is_fraud', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Số tiền (amt)')
axes[0, 1].set_ylabel('Mật độ')
axes[0, 1].legend()

# 3. KDE overlay
df[df[TARGET_COL] == 0]['amt'].plot.kde(ax=axes[1, 0], color='#2ecc71', label='Hợp lệ', bw_method=0.3)
df[df[TARGET_COL] == 1]['amt'].plot.kde(ax=axes[1, 0], color='#e74c3c', label='Gian lận', bw_method=0.3)
axes[1, 0].set_xlim(0, df['amt'].quantile(0.99))
axes[1, 0].set_title('KDE amt — so sánh legit vs fraud', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Số tiền (amt)')
axes[1, 0].legend()

# 4. Histogram log-scale
df['amt_log'] = np.log1p(df['amt'])
for label, color, name in [(0, '#2ecc71', 'Hợp lệ'), (1, '#e74c3c', 'Gian lận')]:
    subset = df[df[TARGET_COL] == label]['amt_log']
    axes[1, 1].hist(subset, bins=80, alpha=0.6, color=color, label=name, density=True)
axes[1, 1].set_title('Phân phối log(amt) — tách theo is_fraud', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('log(1 + amt)')
axes[1, 1].set_ylabel('Mật độ')
axes[1, 1].legend()

fig.suptitle('Phân phối số tiền giao dịch (amt)', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig('02_amt_distribution', fig)
plt.show()

### 4.2 Boxplot `amt` theo `is_fraud`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ["#2ecc71", "#e74c3c"]

# Boxplot scale thường
sns.boxplot(
    data=df, x=TARGET_COL, y='amt',
    hue=TARGET_COL, legend=False,
    palette=colors, ax=axes[0],
    flierprops={'marker': '.', 'markersize': 1, 'alpha': 0.3}
)
axes[0].set_title('Boxplot amt theo is_fraud', fontsize=12, fontweight='bold')
axes[0].set_xlabel('is_fraud')

# Boxplot log-scale
sns.boxplot(
    data=df, x=TARGET_COL, y='amt',
    hue=TARGET_COL, legend=False,
    palette=colors, ax=axes[1],
    flierprops={'marker': '.', 'markersize': 1, 'alpha': 0.3}
)
axes[1].set_yscale('log')
axes[1].set_title('Boxplot amt theo is_fraud (log scale)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('is_fraud')

plt.tight_layout()
save_fig('03_amt_boxplot', fig)
plt.show()


### 4.3 Phân phối theo thời gian

Phân tích số giao dịch và tỷ lệ gian lận theo **giờ trong ngày** và **ngày trong tuần** (derive từ `unix_time`).

In [ ]:
# Derive time features từ unix_time
df['trans_datetime'] = pd.to_datetime(df['unix_time'], unit='s')
df['hour'] = df['trans_datetime'].dt.hour
df['day_of_week'] = df['trans_datetime'].dt.dayofweek  # 0=Monday, 6=Sunday
df['day_name'] = df['trans_datetime'].dt.day_name()

print("✅ Đã tạo các cột thời gian: hour, day_of_week, day_name")
print(f"   Khoảng thời gian: {df['trans_datetime'].min()} → {df['trans_datetime'].max()}")

In [ ]:
# Phân phối theo giờ trong ngày
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Tổng giao dịch theo giờ
hourly_counts = df.groupby('hour').size()
hourly_fraud = df.groupby('hour')[TARGET_COL].mean() * 100

axes[0].bar(hourly_counts.index, hourly_counts.values, color='steelblue', edgecolor='white')
axes[0].set_title('Số giao dịch theo giờ trong ngày', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Giờ')
axes[0].set_ylabel('Số giao dịch')
axes[0].set_xticks(range(24))

# Tỷ lệ gian lận theo giờ
bars = axes[1].bar(hourly_fraud.index, hourly_fraud.values, color='#e74c3c', edgecolor='white')
axes[1].set_title('Tỷ lệ gian lận theo giờ trong ngày', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Giờ')
axes[1].set_ylabel('Tỷ lệ gian lận (%)')
axes[1].set_xticks(range(24))
axes[1].axhline(y=df[TARGET_COL].mean()*100, color='gray', linestyle='--', alpha=0.7, label='Trung bình')
axes[1].legend()

fig.suptitle('Phân tích giao dịch theo giờ trong ngày', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('04_hourly_distribution', fig)
plt.show()

In [ ]:
# Phân phối theo ngày trong tuần
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Tổng giao dịch theo ngày
daily_counts = df.groupby('day_name').size().reindex(day_order)
daily_fraud = df.groupby('day_name')[TARGET_COL].mean().reindex(day_order) * 100

axes[0].bar(range(7), daily_counts.values, color='#3498db', edgecolor='white')
axes[0].set_title('Số giao dịch theo ngày trong tuần', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Ngày')
axes[0].set_ylabel('Số giao dịch')
axes[0].set_xticks(range(7))
axes[0].set_xticklabels(['T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'CN'], fontsize=10)

# Tỷ lệ gian lận theo ngày
axes[1].bar(range(7), daily_fraud.values, color='#e74c3c', edgecolor='white')
axes[1].set_title('Tỷ lệ gian lận theo ngày trong tuần', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Ngày')
axes[1].set_ylabel('Tỷ lệ gian lận (%)')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(['T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'CN'], fontsize=10)
axes[1].axhline(y=df[TARGET_COL].mean()*100, color='gray', linestyle='--', alpha=0.7, label='Trung bình')
axes[1].legend()

fig.suptitle('Phân tích giao dịch theo ngày trong tuần', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('05_daily_distribution', fig)
plt.show()

### 4.4 Tỷ lệ gian lận theo Category (loại hình mua sắm)

Sắp xếp giảm dần theo tỷ lệ gian lận để nhận diện các loại hình có rủi ro cao.

In [ ]:
# Tỷ lệ gian lận theo category
cat_fraud = df.groupby('category').agg(
    total=('is_fraud', 'count'),
    n_fraud=('is_fraud', 'sum')
).assign(fraud_rate=lambda x: x['n_fraud'] / x['total'] * 100)
cat_fraud = cat_fraud.sort_values('fraud_rate', ascending=True)  # ascending for barh

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(
    cat_fraud.index, cat_fraud['fraud_rate'],
    color=plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, len(cat_fraud))),
    edgecolor='white'
)

# Thêm label giá trị
for bar, rate in zip(bars, cat_fraud['fraud_rate']):
    ax.text(
        bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
        f'{rate:.2f}%', va='center', fontsize=9
    )

ax.axvline(x=df[TARGET_COL].mean()*100, color='gray', linestyle='--', alpha=0.7, label='Tỷ lệ trung bình')
ax.set_title('Tỷ lệ gian lận theo Category — sắp xếp giảm dần', fontsize=13, fontweight='bold')
ax.set_xlabel('Tỷ lệ gian lận (%)')
ax.legend()
plt.tight_layout()
save_fig('06_fraud_rate_by_category', fig)
plt.show()

### 4.5 Tỷ lệ gian lận theo Gender

In [ ]:
# Tỷ lệ gian lận theo gender
gender_fraud = df.groupby('gender').agg(
    total=('is_fraud', 'count'),
    n_fraud=('is_fraud', 'sum')
).assign(fraud_rate=lambda x: x['n_fraud'] / x['total'] * 100)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Số lượng theo gender
colors_gender = ['#3498db', '#e91e63']
bars1 = axes[0].bar(gender_fraud.index, gender_fraud['total'], color=colors_gender, edgecolor='white')
for bar, count in zip(bars1, gender_fraud['total']):
    axes[0].text(
        bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
        f'{count:,}', ha='center', va='bottom', fontweight='bold'
    )
axes[0].set_title('Tổng giao dịch theo Gender', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Số giao dịch')

# Tỷ lệ gian lận theo gender
bars2 = axes[1].bar(gender_fraud.index, gender_fraud['fraud_rate'], color=colors_gender, edgecolor='white')
for bar, rate in zip(bars2, gender_fraud['fraud_rate']):
    axes[1].text(
        bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
        f'{rate:.2f}%', ha='center', va='bottom', fontweight='bold'
    )
axes[1].set_title('Tỷ lệ gian lận theo Gender', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Tỷ lệ gian lận (%)')

plt.tight_layout()
save_fig('07_fraud_rate_by_gender', fig)
plt.show()

---
## 5. Phân tích tương quan

Correlation heatmap cho các biến numerical, nhận diện biến nào có tương quan đáng chú ý với `is_fraud`.

In [ ]:
# Correlation heatmap
corr_cols = [c for c in NUMERICAL_COLS if c in df.columns] + [TARGET_COL]
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # chỉ hiện nửa dưới
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.3f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Correlation Heatmap — Numerical Features + is_fraud', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('08_correlation_heatmap', fig)
plt.show()

In [ ]:
# Tương quan với is_fraud — sắp xếp theo giá trị tuyệt đối
corr_with_fraud = corr_matrix[TARGET_COL].drop(TARGET_COL).sort_values(key=abs, ascending=False)

print("📊 Tương quan của các biến numerical với is_fraud:")
print("   (Sắp xếp theo giá trị tuyệt đối giảm dần)\n")

for feat, val in corr_with_fraud.items():
    indicator = "🔴" if abs(val) > 0.1 else "🟡" if abs(val) > 0.05 else "⚪"
    print(f"   {indicator} {feat:15s}: {val:+.4f}")

print("\n📝 Nhận xét:")
print("   - Các biến numerical có tương quan thấp với is_fraud (đặc trưng của bài toán fraud).")
print("   - Biến 'amt' thường có tương quan dương nhẹ — giao dịch gian lận có xu hướng giá trị cao hơn.")
print("   - Tương quan thấp KHÔNG có nghĩa là biến không hữu ích — quan hệ phi tuyến sẽ được")
print("     mô hình ML khai thác.")

---
## 6. Age Feature — Phân tích tuổi khách hàng

Tính tuổi khách hàng từ cột `dob` (ngày sinh) và phân tích phân phối theo `is_fraud`.

In [ ]:
# Tính tuổi từ dob
df['dob'] = pd.to_datetime(df['dob'])
# Lấy ngày giao dịch để tính tuổi chính xác tại thời điểm giao dịch
df['age'] = (df['trans_datetime'] - df['dob']).dt.days // 365

print(f"📊 Thống kê tuổi khách hàng:")
print(f"   Min: {df['age'].min()}, Max: {df['age'].max()}, Mean: {df['age'].mean():.1f}, Median: {df['age'].median():.0f}")
print(f"\n   Tuổi trung bình (legit):  {df[df[TARGET_COL]==0]['age'].mean():.1f}")
print(f"   Tuổi trung bình (fraud):  {df[df[TARGET_COL]==1]['age'].mean():.1f}")

In [ ]:
# Histogram phân phối tuổi — tách theo is_fraud
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram overlay
for label, color, name in [(0, '#2ecc71', 'Hợp lệ'), (1, '#e74c3c', 'Gian lận')]:
    subset = df[df[TARGET_COL] == label]['age']
    axes[0].hist(subset, bins=50, alpha=0.6, color=color, label=name, density=True)
axes[0].set_title('Phân phối tuổi — tách theo is_fraud', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Tuổi')
axes[0].set_ylabel('Mật độ')
axes[0].legend()

# Tỷ lệ gian lận theo nhóm tuổi
df['age_group'] = pd.cut(df['age'], bins=range(0, 105, 10), right=False)
age_fraud = df.groupby('age_group', observed=True)[TARGET_COL].mean() * 100

age_labels = [str(g) for g in age_fraud.index]
bars = axes[1].bar(range(len(age_fraud)), age_fraud.values, color='#e74c3c', edgecolor='white', alpha=0.85)
axes[1].set_xticks(range(len(age_fraud)))
axes[1].set_xticklabels(age_labels, rotation=45, ha='right', fontsize=9)
axes[1].set_title('Tỷ lệ gian lận theo nhóm tuổi', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Nhóm tuổi')
axes[1].set_ylabel('Tỷ lệ gian lận (%)')
axes[1].axhline(y=df[TARGET_COL].mean()*100, color='gray', linestyle='--', alpha=0.7, label='Trung bình')
axes[1].legend()

# Thêm giá trị trên bar
for bar, rate in zip(bars, age_fraud.values):
    axes[1].text(
        bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
        f'{rate:.1f}%', ha='center', va='bottom', fontsize=8
    )

fig.suptitle('Phân tích tuổi khách hàng', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('09_age_distribution', fig)
plt.show()

---
## 7. Data Profiling tự động (ydata-profiling)

Sinh báo cáo profiling đầy đủ: missing values, distributions, correlations, warnings (skewness, high cardinality...).

> ⚠️ **Lưu ý**: Cell này có thể chạy lâu (5-15 phút) do dataset lớn. Có thể dùng `minimal=True` để tăng tốc.

In [ ]:
from ydata_profiling import ProfileReport

# Chọn các cột chính để profiling (bỏ cột phụ đã derive)
profile_cols = [c for c in df.columns if c not in ['_source', 'amt_log', 'trans_datetime', 'hour', 'day_of_week', 'day_name', 'age', 'age_group']]
df_profile = df[profile_cols]

print(f"📊 Đang tạo profiling report cho {df_profile.shape[0]:,} rows × {df_profile.shape[1]} columns...")
print("   Có thể mất vài phút...\n")

profile = ProfileReport(
    df_profile,
    title="Sparkov Fraud Detection — Data Profiling Report",
    explorative=True,
    minimal=False,
    progress_bar=True,
)

# Lưu HTML report
report_path = PROFILING_DIR / "sparkov_profile_report.html"
profile.to_file(report_path)
print(f"\n✅ Report đã lưu tại: {report_path}")
print(f"   Mở file HTML trên browser để xem đầy đủ.")

---
## Tổng kết

### Data Analysis Key Findings

- **Dữ liệu sạch**: Không có missing values, không có duplicate rows — đặc trưng của dữ liệu mô phỏng Sparkov.
- **Mất cân bằng nghiêm trọng**: Lớp gian lận chiếm tỷ lệ rất nhỏ so với hợp lệ → cần xử lý mất cân bằng ở bước tiền xử lý.
- **Phân phối `amt`**: Giao dịch gian lận có xu hướng giá trị cao hơn, phân phối lệch phải mạnh.
- **Thời gian**: Giao dịch gian lận tập trung nhiều hơn vào ban đêm/rạng sáng (giờ ít giao dịch).
- **Category**: Một số loại hình mua sắm có tỷ lệ gian lận cao hơn đáng kể.
- **Tương quan tuyến tính thấp**: Các biến numerical ít tương quan tuyến tính với `is_fraud` → mô hình cần khai thác quan hệ phi tuyến.
- **Tuổi**: Người cao tuổi có xu hướng bị gian lận nhiều hơn.

### Insights & Next Steps

- **Bước tiếp theo**: Tiền xử lý dữ liệu — encoding categorical variables, feature engineering (hour, age, distance...), xử lý mất cân bằng (SMOTE, SMOTE-ENN, class weighting).
- **Cân nhắc**: Feature `amt`, `hour`, `age`, `category` là các ứng viên mạnh cho mô hình phân loại.